# 🎭 DeepGuard AI — Deepfake Dataset Generation v2
### Face-swap dataset using InsightFace inswapper_128
**Goal:** Generate 6,000+ face-swapped images → save to Google Drive → publish on HuggingFace

**Runtime:** T4 GPU | **Est. time:** 2-3 hours | **Images saved to Google Drive in real-time**

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install insightface onnxruntime-gpu huggingface_hub kaggle opencv-python-headless tqdm datasets -q

import torch
print(f'✅ CUDA available: {torch.cuda.is_available()}')
print('✅ All dependencies installed')

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive (saves progress even if session expires)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# All outputs go to Google Drive
BASE_DIR  = '/content/drive/MyDrive/deepguard_dataset'
OUT_REAL  = f'{BASE_DIR}/real'
OUT_FAKE  = f'{BASE_DIR}/fake'

os.makedirs(OUT_REAL, exist_ok=True)
os.makedirs(OUT_FAKE, exist_ok=True)

# Count already generated images (resume support)
already_done = len([f for f in os.listdir(OUT_FAKE) if f.endswith('.jpg')])

print(f'✅ Google Drive mounted')
print(f'✅ Output folder: {BASE_DIR}')
print(f'📊 Already generated: {already_done} images (resuming from here)')

In [ ]:
# ============================================================
# CELL 3: Upload kaggle.json and download real face images
# ============================================================
from google.colab import files

print('📁 Upload your kaggle.json file...')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set')

print('⬇️  Downloading dataset (~5 mins)...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip
print('✅ Dataset downloaded!')

In [ ]:
# ============================================================
# CELL 4: Find real face images and prepare pairs
# ============================================================
import glob
import random

real_imgs = glob.glob('/content/data/**/real/**/*.jpg', recursive=True) + \
            glob.glob('/content/data/**/real/**/*.png', recursive=True)

random.seed(42)
random.shuffle(real_imgs)

# Use 20000 images to ensure enough successful swaps after failures
selected = real_imgs[:20000]
sources  = selected[:10000]
targets  = selected[10000:]

print(f'✅ Real images found: {len(real_imgs)}')
print(f'✅ Source faces: {len(sources)}')
print(f'✅ Target faces: {len(targets)}')

In [ ]:
# ============================================================
# CELL 5: Download InsightFace models
# ============================================================
import os

os.makedirs('/content/models', exist_ok=True)
os.makedirs('/root/.insightface/models', exist_ok=True)

# Download buffalo_l face detection model
print('⬇️  Downloading buffalo_l model...')
!wget -q "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip" \
     -O /tmp/buffalo_l.zip
!unzip -q /tmp/buffalo_l.zip -d /root/.insightface/models/buffalo_l/
print('✅ buffalo_l downloaded')

# Download inswapper model
print('⬇️  Downloading inswapper_128.onnx...')
!wget -q "https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx" \
     -O /content/models/inswapper_128.onnx
INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
print(f'✅ inswapper downloaded: {os.path.exists(INSWAPPER_PATH)}')

# Setup InsightFace with SMALLER detection size = catches more faces
import insightface
from insightface.app import FaceAnalysis

face_app = FaceAnalysis(
    name='buffalo_l',
    root='/root/.insightface',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
# 320x320 instead of 640x640 — detects more faces including smaller/angled ones
face_app.prepare(ctx_id=0, det_size=(320, 320))

swapper = insightface.model_zoo.get_model(
    INSWAPPER_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

print('✅ Face analyzer ready (det_size=320x320)')
print('✅ Face swapper ready')

In [ ]:
# ============================================================
# CELL 6: Generate deepfake images (saves to Google Drive)
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm
import csv
import os

TARGET_COUNT  = 6000   # Total fake images to generate
SAVE_CSV_EVERY = 100   # Save metadata CSV every N images

# Resume from previous session
success_count = already_done
fail_count    = 0
metadata      = []

# Load existing metadata if resuming
csv_path = f'{BASE_DIR}/metadata.csv'
if os.path.exists(csv_path) and already_done > 0:
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        metadata = list(reader)
    print(f'📂 Resuming from {already_done} images, {len(metadata)} metadata rows loaded')

remaining = TARGET_COUNT - success_count
print(f'🎭 Generating {remaining} more deepfake images (target: {TARGET_COUNT} total)...')

pair_iter = list(zip(sources, targets))
# Skip already processed pairs
pair_iter = pair_iter[already_done * 2:] if already_done > 0 else pair_iter

for src_path, tgt_path in tqdm(pair_iter, total=len(pair_iter)):
    if success_count >= TARGET_COUNT:
        break
    try:
        src_img = cv2.imread(src_path)
        tgt_img = cv2.imread(tgt_path)
        if src_img is None or tgt_img is None:
            fail_count += 1
            continue

        src_faces = face_app.get(src_img)
        tgt_faces = face_app.get(tgt_img)

        if not src_faces or not tgt_faces:
            fail_count += 1
            continue

        # Face swap
        result = tgt_img.copy()
        result = swapper.get(result, tgt_faces[0], src_faces[0], paste_back=True)

        # Save to Google Drive
        fake_name = f'fake_{success_count:05d}.jpg'
        real_name = f'real_{success_count:05d}.jpg'
        cv2.imwrite(f'{OUT_FAKE}/{fake_name}', result, [cv2.IMWRITE_JPEG_QUALITY, 95])
        cv2.imwrite(f'{OUT_REAL}/{real_name}', tgt_img,  [cv2.IMWRITE_JPEG_QUALITY, 95])

        metadata.append({
            'id': success_count,
            'fake_file': f'fake/{fake_name}',
            'real_file': f'real/{real_name}',
            'method': 'inswapper_128',
            'source_identity': os.path.basename(src_path),
            'target_face': os.path.basename(tgt_path),
            'label_fake': 1,
            'label_real': 0,
        })
        success_count += 1

        # Save CSV periodically so progress is never lost
        if success_count % SAVE_CSV_EVERY == 0:
            with open(csv_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=metadata[0].keys())
                writer.writeheader()
                writer.writerows(metadata)
            print(f'  ✅ {success_count}/{TARGET_COUNT} generated | failed: {fail_count} | CSV saved')

    except Exception as e:
        fail_count += 1
        continue

# Final CSV save
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=metadata[0].keys())
    writer.writeheader()
    writer.writerows(metadata)

print(f'\n✅ Generation complete!')
print(f'   Fake images: {success_count}')
print(f'   Real images: {success_count}')
print(f'   Failed: {fail_count}')
print(f'   Saved to: {BASE_DIR}')

In [ ]:
# ============================================================
# CELL 7: Save README and verify dataset
# ============================================================
import os

real_count = len([f for f in os.listdir(OUT_REAL) if f.endswith('.jpg')])
fake_count = len([f for f in os.listdir(OUT_FAKE) if f.endswith('.jpg')])

readme = f"""# DeepGuard Deepfake Dataset

A paired deepfake dataset for training and benchmarking deepfake detection models.
Generated as part of the DeepGuard AI Final Year Project (2026).

## Dataset Statistics
- **Fake images:** {fake_count} (face-swapped using InsightFace inswapper_128)
- **Real images:** {real_count} (original paired faces)
- **Total:** {fake_count + real_count} images
- **Format:** JPEG, high quality (95%)
- **Generation method:** InsightFace inswapper_128 (ONNX runtime)

## Why This Dataset is Unique
- **Paired structure:** Every fake has a matching real image of the same target person
- **Modern method:** inswapper_128 powers most 2024-2026 real-world deepfake apps
- **Traceable metadata:** Source identity, target face, and method per image
- **Open access:** No sign-up required, instant `load_dataset()` access

## Structure
```
deepguard-dataset/
├── real/           # Real face images (label=0)
├── fake/           # Face-swapped deepfakes (label=1)
└── metadata.csv    # id, paths, method, source, target, labels
```

## Quick Start
```python
from datasets import load_dataset
ds = load_dataset('Sowaiba-01/deepguard-dataset')
```

## Detection Model
Trained EfficientNet-B4 detection model also available:
```python
from huggingface_hub import hf_hub_download
model_path = hf_hub_download('Sowaiba-01/deepguard-ai', 'efficientnet_b4_deepguard.pth')
```
Validation accuracy: **99.78%** on 140K Real & Fake Faces dataset.

## Citation
```
@dataset{{deepguard2026,
  author = {{Arshad, Sowaiba}},
  title  = {{DeepGuard Deepfake Dataset}},
  year   = {{2026}},
  url    = {{https://huggingface.co/datasets/Sowaiba-01/deepguard-dataset}}
}}
```
"""

with open(f'{BASE_DIR}/README.md', 'w') as f:
    f.write(readme)

print(f'✅ README saved')
print(f'\n📊 Final dataset:')
print(f'   Real:  {real_count} images')
print(f'   Fake:  {fake_count} images')
print(f'   Total: {real_count + fake_count} images')
print(f'   CSV rows: {len(metadata)}')

In [ ]:
# ============================================================
# CELL 8: Push dataset to HuggingFace
# ============================================================
# Get token from: https://huggingface.co/settings/tokens (Write permission)
HF_TOKEN = 'your_hf_token_here'   # ← PASTE YOUR HF TOKEN HERE
HF_REPO  = 'Sowaiba-01/deepguard-dataset'

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type='dataset', exist_ok=True)
print(f'✅ Repo created: {HF_REPO}')

api.upload_file(path_or_fileobj=f'{BASE_DIR}/metadata.csv',
                path_in_repo='metadata.csv', repo_id=HF_REPO, repo_type='dataset')
api.upload_file(path_or_fileobj=f'{BASE_DIR}/README.md',
                path_in_repo='README.md', repo_id=HF_REPO, repo_type='dataset')
print('✅ Metadata + README uploaded')

print('⬆️  Uploading fake images (this takes a while)...')
api.upload_folder(folder_path=OUT_FAKE, path_in_repo='fake',
                  repo_id=HF_REPO, repo_type='dataset')
print('✅ Fake images uploaded')

print('⬆️  Uploading real images...')
api.upload_folder(folder_path=OUT_REAL, path_in_repo='real',
                  repo_id=HF_REPO, repo_type='dataset')
print('✅ Real images uploaded')

print(f'\n🎉 Dataset live at: https://huggingface.co/datasets/{HF_REPO}')

In [ ]:
# ============================================================
# CELL 9: Push trained detection model to HuggingFace
# ============================================================
from google.colab import files

HF_MODEL_REPO = 'Sowaiba-01/deepguard-ai'

print('📁 Upload efficientnet_b4_deepguard.pth from your computer...')
uploaded = files.upload()

api.create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj='efficientnet_b4_deepguard.pth',
    path_in_repo='efficientnet_b4_deepguard.pth',
    repo_id=HF_MODEL_REPO,
    repo_type='model',
)
print(f'✅ Model live at: https://huggingface.co/{HF_MODEL_REPO}')